# Notebook 05: TensorFlow/Keras MLP

**Objective:** Test whether a neural tabular model beats the selected LightGBM baseline. Train on dev (2019 to 2021), select on val (2022). Holdout rows are never loaded.

**Decision rule, fixed before any result here.** The MLP is retained for a target only if all three hold:
1. Its val PR-AUC is higher than LightGBM's in each of 3 training seeds.
2. The paired bootstrap 95% interval for the PR-AUC difference (LightGBM vs the mean prediction of the 3 seeds) excludes zero.
3. The mean difference is at least 0.002.

Otherwise it is recorded as a rejected challenger. Budget: at most 4 architecture or hyperparameter settings per target, then 3 seeds for the best one. Preprocessing follows the Notebook 03 rule: median imputation fitted on dev, no missingness indicators, no class weighting (Notebook 04).

In [1]:
import sys, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")
print("python", sys.version.split()[0])

import tensorflow as tf
print("tensorflow", tf.__version__, "| keras", tf.keras.__version__)
print("devices:", [d.device_type for d in tf.config.list_physical_devices()])

targets = ["adverse_outcome", "readmission_30d"]

# ---- data: holdout rows are filtered out at read time ----
mr = pd.read_parquet("../data/interim/model_ready.parquet", filters=[("split", "!=", "holdout")])
feat_tbl = pd.read_csv("../reports/tables/feature_summary.csv")
features = feat_tbl["feature"].tolist()
cat_features = feat_tbl.loc[feat_tbl["kind"] == "categorical", "feature"].tolist()
binary_features = feat_tbl.loc[feat_tbl["kind"] == "binary", "feature"].tolist()
numeric_features = feat_tbl.loc[feat_tbl["kind"] == "numeric", "feature"].tolist()
assert not {"arrival_day", "arrival_month", "arrival_day_of_week", "arrival_year"} & set(features)

dev = mr[mr["split"] == "dev"].reset_index(drop=True)
val = mr[mr["split"] == "val"].reset_index(drop=True)
del mr

# =============== SHARED FUNCTIONS (copied from Notebook 04) ===============
def ece_score(y, p, n_bins=10):
    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
    return sum((bins == b).mean() * abs(p[bins == b].mean() - y[bins == b].mean())
               for b in range(n_bins) if (bins == b).any())

def paired_boot(y, p_a, p_b, n=200, seed=0):
    rs = np.random.default_rng(seed)
    y, p_a, p_b = np.asarray(y), np.asarray(p_a), np.asarray(p_b)
    diffs = []
    for _ in range(n):
        i = rs.integers(0, len(y), len(y))
        diffs.append(average_precision_score(y[i], p_b[i]) - average_precision_score(y[i], p_a[i]))
    return np.percentile(diffs, [2.5, 50, 97.5]).round(4)

K_BY_TARGET = {"adverse_outcome": [0.05, 0.10, 0.20, 0.30, 0.40],
               "readmission_30d": [0.01, 0.05, 0.10, 0.20]}

def rank_table(y, score, ks, seed=42):
    y, score = np.asarray(y), np.asarray(score)
    jitter = np.random.default_rng(seed).random(len(score)) * 1e-9
    order = np.argsort(-(score + jitter))
    total, prev, rows = int(y.sum()), y.mean(), []
    for k in ks:
        n = int(round(len(y) * k))
        hits = int(y[order[:n]].sum())
        rows.append({"K": k, "n_review": n, "events_captured": hits, "total_events": total,
                     "recall": hits / total, "precision": hits / n,
                     "lift": (hits / n) / prev, "false_positives": n - hits})
    return pd.DataFrame(rows)
# ==========================================================================

# ---- saved LightGBM val predictions from Notebook 04 ----
vp = pd.read_parquet("../data/interim/val_predictions_classical.parquet")
assert (vp["encounter_id"].values == val["encounter_id"].values).all(), "row order differs"
for t in targets:
    p = vp[f"lightgbm_{t}"].values
    print(t, "| LightGBM val PR-AUC from saved file:", round(average_precision_score(val[t], p), 4))

print()
print("features:", len(features), "| numeric", len(numeric_features), "| binary", len(binary_features),
      "| categorical", len(cat_features))
print("dev:", dev.shape, "years", sorted(int(y) for y in dev["arrival_year"].unique()))
print("val:", val.shape, "years", sorted(int(y) for y in val["arrival_year"].unique()))
print("splits present:", sorted(set(dev["split"]) | set(val["split"])))

python 3.13.5
tensorflow 2.20.0 | keras 3.11.3
devices: ['CPU']
adverse_outcome | LightGBM val PR-AUC from saved file: 0.7553
readmission_30d | LightGBM val PR-AUC from saved file: 0.2138

features: 90 | numeric 53 | binary 30 | categorical 7
dev: (400365, 105) years [2019, 2020, 2021]
val: (133853, 105) years [2022]
splits present: ['dev', 'val']


In [3]:
from tensorflow import keras
from tensorflow.keras import layers

# ---- preprocessing, fitted on dev only ----
num_med  = dev[numeric_features].median()
filled   = dev[numeric_features].fillna(num_med)
num_mean = filled.mean()
num_std  = filled.std().replace(0, 1.0)

cat_maps = {c: {v: i + 1 for i, v in enumerate(sorted(dev[c].unique()))} for c in cat_features}

def make_inputs(d):
    Xn = ((d[numeric_features].fillna(num_med) - num_mean) / num_std).astype("float32").values
    Xb = d[binary_features].astype("float32").values
    X = {"num_bin": np.hstack([Xn, Xb])}
    for c in cat_features:
        X[c] = d[c].map(cat_maps[c]).fillna(0).astype("int32").values.reshape(-1, 1)
    return X

Xd, Xv = make_inputs(dev), make_inputs(val)
assert not np.isnan(Xd["num_bin"]).any() and not np.isnan(Xv["num_bin"]).any()
n_dense = Xd["num_bin"].shape[1]
print("dense inputs:", n_dense, "| unseen category codes in val:", {c: int((Xv[c] == 0).sum()) for c in cat_features})

# ---- model ----
def build_mlp(hidden=(256, 128), dropout=0.2, lr=1e-3, emb_max=8):
    dense_in = keras.Input(shape=(n_dense,), name="num_bin")
    inputs, parts = [dense_in], [dense_in]
    for c in cat_features:
        card = len(cat_maps[c]) + 1
        dim = min(emb_max, (card + 1) // 2)
        i = keras.Input(shape=(1,), dtype="int32", name=c)
        parts.append(layers.Flatten()(layers.Embedding(card, dim)(i)))
        inputs.append(i)
    x = layers.Concatenate()(parts)
    for h in hidden:
        x = layers.Dense(h)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model(inputs, out)
    m.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(curve="PR", name="auc_pr")])
    return m

# ---- timing and sanity run: 2 epochs, adverse_outcome only ----
t = "adverse_outcome"
keras.utils.set_random_seed(0)
model = build_mlp()
print("parameters:", model.count_params())

t0 = time.time()
model.fit(Xd, dev[t].values.astype("float32"),
          validation_data=(Xv, val[t].values.astype("float32")),
          epochs=2, batch_size=1024, verbose=2)
print("seconds for 2 epochs:", round(time.time() - t0))

p = model.predict(Xv, batch_size=4096, verbose=0).ravel()
print("val PR-AUC (sklearn):", round(average_precision_score(val[t], p), 4),
      "| ROC-AUC:", round(roc_auc_score(val[t], p), 4),
      "| prevalence:", round(val[t].mean(), 4))

dense inputs: 83 | unseen category codes in val: {'city': 0, 'urban_rural': 0, 'hospital_type': 0, 'gender': 0, 'insurance_type': 0, 'arrival_mode': 0, 'chief_complaint': 0}
parameters: 64283
Epoch 1/2
391/391 - 10s - 25ms/step - auc_pr: 0.7188 - loss: 0.5557 - val_auc_pr: 0.7460 - val_loss: 0.5340
Epoch 2/2
391/391 - 4s - 10ms/step - auc_pr: 0.7418 - loss: 0.5346 - val_auc_pr: 0.7476 - val_loss: 0.5298
seconds for 2 epochs: 14
val PR-AUC (sklearn): 0.7477 | ROC-AUC: 0.798 | prevalence: 0.408


In [4]:
SETTINGS = {1: dict(hidden=(256, 128), dropout=0.2, lr=1e-3),
            2: dict(hidden=(512, 256, 128), dropout=0.3, lr=1e-3),
            3: dict(hidden=(128, 64), dropout=0.1, lr=1e-3),
            4: dict(hidden=(256, 128), dropout=0.2, lr=3e-4)}

mlp_preds, mlp_log = {}, []

def run_mlp(setting, t, seed):
    keras.utils.set_random_seed(seed)
    m = build_mlp(**SETTINGS[setting])
    cb = keras.callbacks.EarlyStopping(monitor="val_auc_pr", mode="max", patience=5,
                                       restore_best_weights=True)
    t0 = time.time()
    h = m.fit(Xd, dev[t].values.astype("float32"),
              validation_data=(Xv, val[t].values.astype("float32")),
              epochs=60, batch_size=1024, callbacks=[cb], verbose=0)
    p = m.predict(Xv, batch_size=4096, verbose=0).ravel()
    best_ep = int(np.argmax(h.history["val_auc_pr"])) + 1
    y = val[t].values
    mlp_preds[(setting, t, seed)] = p
    mlp_log.append({"setting": setting, "target": t, "seed": seed, "best_epoch": best_ep,
                    "epochs_run": len(h.history["loss"]),
                    "pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p),
                    "brier": brier_score_loss(y, p), "ece": ece_score(y, p),
                    "secs": round(time.time() - t0)})
    return p

for t in targets:
    run_mlp(1, t, 0)

lgb_ref = {t: average_precision_score(val[t], vp[f"lightgbm_{t}"].values) for t in targets}
log = pd.DataFrame(mlp_log)
log["lightgbm_pr_auc"] = log["target"].map(lgb_ref)
print(log.round(4).to_string(index=False))

 setting          target  seed  best_epoch  epochs_run  pr_auc  roc_auc  brier    ece  secs  lightgbm_pr_auc
       1 adverse_outcome     0           7          12  0.7489   0.7987 0.1762 0.0034    56           0.7553
       1 readmission_30d     0           4           9  0.2097   0.5999 0.1159 0.0299    45           0.2138


In [5]:
for s in [2, 3, 4]:
    for t in targets:
        run_mlp(s, t, 0)
        print("done: setting", s, t)

log = pd.DataFrame(mlp_log)
log["lightgbm_pr_auc"] = log["target"].map(lgb_ref)
log["gap_vs_lightgbm"] = log["pr_auc"] - log["lightgbm_pr_auc"]
print()
print(log.sort_values(["target", "setting"]).round(4).to_string(index=False))

best_setting = {t: int(log[log["target"] == t].sort_values("pr_auc").iloc[-1]["setting"]) for t in targets}
print()
print("best setting by val PR-AUC:", best_setting)

done: setting 2 adverse_outcome
done: setting 2 readmission_30d
done: setting 3 adverse_outcome
done: setting 3 readmission_30d
done: setting 4 adverse_outcome
done: setting 4 readmission_30d

 setting          target  seed  best_epoch  epochs_run  pr_auc  roc_auc  brier    ece  secs  lightgbm_pr_auc  gap_vs_lightgbm
       1 adverse_outcome     0           7          12  0.7489   0.7987 0.1762 0.0034    56           0.7553          -0.0063
       2 adverse_outcome     0           5          10  0.7489   0.7990 0.1762 0.0075    80           0.7553          -0.0064
       3 adverse_outcome     0           6          11  0.7487   0.7986 0.1764 0.0087    55           0.7553          -0.0065
       4 adverse_outcome     0          12          17  0.7494   0.7990 0.1761 0.0047    88           0.7553          -0.0058
       1 readmission_30d     0           4           9  0.2097   0.5999 0.1159 0.0299    45           0.2138          -0.0041
       2 readmission_30d     0           5         

In [6]:
for t in targets:
    s = best_setting[t]
    for seed in [1, 2]:
        run_mlp(s, t, seed)
        print("done: setting", s, t, "seed", seed)

log = pd.DataFrame(mlp_log)
lgb_p = {t: vp[f"lightgbm_{t}"].values for t in targets}
lr_p  = {t: vp[f"logreg_{t}"].values for t in targets}

verdict = {}
print()
for t in targets:
    s = best_setting[t]
    seeds = log[(log["target"] == t) & (log["setting"] == s)].sort_values("seed")
    ens = np.mean([mlp_preds[(s, t, sd)] for sd in seeds["seed"]], axis=0)
    prs = seeds["pr_auc"].values
    mean_gap = prs.mean() - lgb_ref[t]
    ci_lgb = paired_boot(val[t], lgb_p[t], ens)
    ci_lr = paired_boot(val[t], lr_p[t], ens)
    c1 = bool((prs > lgb_ref[t]).all())
    c2 = bool(ci_lgb[0] > 0)
    c3 = bool(mean_gap >= 0.002)
    verdict[t] = (c1, c2, c3)
    print(t, "| setting", s)
    print("  MLP PR-AUC by seed:", prs.round(4), "| mean:", round(prs.mean(), 4), "| SD:", round(prs.std(ddof=1), 4))
    print("  ensemble of 3 seeds PR-AUC:", round(average_precision_score(val[t], ens), 4),
          "| LightGBM:", round(lgb_ref[t], 4))
    print("  mean gap vs LightGBM:", round(mean_gap, 4))
    print("  MLP-ensemble minus LightGBM [2.5%, median, 97.5%]:", ci_lgb)
    print("  MLP-ensemble minus LogReg   [2.5%, median, 97.5%]:", ci_lr)
    print("  rule 1 (above LightGBM in every seed):", c1, "| rule 2 (interval above zero):", c2,
          "| rule 3 (mean gap >= 0.002):", c3, "->", "RETAIN" if all(verdict[t]) else "REJECT")
    print()

done: setting 4 adverse_outcome seed 1
done: setting 4 adverse_outcome seed 2
done: setting 2 readmission_30d seed 1
done: setting 2 readmission_30d seed 2

adverse_outcome | setting 4
  MLP PR-AUC by seed: [0.7494 0.7494 0.7495] | mean: 0.7494 | SD: 0.0
  ensemble of 3 seeds PR-AUC: 0.7501 | LightGBM: 0.7553
  mean gap vs LightGBM: -0.0058
  MLP-ensemble minus LightGBM [2.5%, median, 97.5%]: [-0.0059 -0.0051 -0.0044]
  MLP-ensemble minus LogReg   [2.5%, median, 97.5%]: [0.003  0.0037 0.0044]
  rule 1 (above LightGBM in every seed): False | rule 2 (interval above zero): False | rule 3 (mean gap >= 0.002): False -> REJECT

readmission_30d | setting 2
  MLP PR-AUC by seed: [0.2101 0.2102 0.2115] | mean: 0.2106 | SD: 0.0008
  ensemble of 3 seeds PR-AUC: 0.2123 | LightGBM: 0.2138
  mean gap vs LightGBM: -0.0032
  MLP-ensemble minus LightGBM [2.5%, median, 97.5%]: [-0.0029 -0.0015  0.0001]
  MLP-ensemble minus LogReg   [2.5%, median, 97.5%]: [-0.0035 -0.0025 -0.0011]
  rule 1 (above LightGB

In [7]:
tables = Path("../reports/tables"); tables.mkdir(parents=True, exist_ok=True)

log = pd.DataFrame(mlp_log)
log["lightgbm_pr_auc"] = log["target"].map(lgb_ref)
log["gap_vs_lightgbm"] = log["pr_auc"] - log["lightgbm_pr_auc"]
log["hidden"] = log["setting"].map(lambda s: str(SETTINGS[s]["hidden"]))
log["dropout"] = log["setting"].map(lambda s: SETTINGS[s]["dropout"])
log["lr"] = log["setting"].map(lambda s: SETTINGS[s]["lr"])
log.round(4).to_csv(tables / "mlp_runs.csv", index=False)

# 3-seed ensemble predictions for later notebooks (local only, val rows only)
mp = val[["encounter_id"] + targets].copy()
mp["split"] = "val"
for t in targets:
    s = best_setting[t]
    mp[f"mlp_{t}"] = np.mean([mlp_preds[(s, t, sd)] for sd in [0, 1, 2]], axis=0)
mp.to_parquet("../data/interim/val_predictions_mlp.parquet", index=False)

print("mlp_runs.csv rows:", len(log), "| val predictions:", mp.shape, "| splits:", sorted(mp["split"].unique()))
for t in targets:
    print(t, "| ensemble PR-AUC from saved file:", round(average_precision_score(mp[t], mp[f"mlp_{t}"]), 4))

for f in ["tables/mlp_runs.csv"]:
    p = Path("../reports") / f
    print(f, "| KB:", round(p.stat().st_size / 1e3, 1) if p.exists() else "MISSING")

mlp_runs.csv rows: 12 | val predictions: (133853, 6) | splits: ['val']
adverse_outcome | ensemble PR-AUC from saved file: 0.7501
readmission_30d | ensemble PR-AUC from saved file: 0.2123
tables/mlp_runs.csv | KB: 1.3


## Setup
Train on dev (2019 to 2021, 400,365 rows), evaluate on val (2022, 133,853 rows). Holdout rows were filtered out at read time. 90 features: 53 numeric (median fill and standardisation fitted on dev), 30 binary, 7 categorical (embeddings up to 8 dimensions, index 0 for unseen categories; none occur in val). Two hidden layers or three, batch normalisation, ReLU, dropout, sigmoid output. Unweighted binary cross-entropy, Adam, batch size 1024, up to 60 epochs, early stopping on val PR-AUC with patience 5 and best-epoch weights restored.

## Decision rule (written before any result)
Retain the MLP for a target only if (1) val PR-AUC beats LightGBM in each of 3 seeds, (2) the paired bootstrap 95% interval for the 3-seed ensemble minus LightGBM excludes zero, (3) the mean PR-AUC gap is at least +0.002. Budget: 4 settings per target, then 3 seeds of the best one.

## Settings tried (seed 0, reports/tables/mlp_runs.csv)
| Setting | Hidden | Dropout | LR | adverse_outcome PR-AUC | readmission_30d PR-AUC |
|---|---|---|---|---:|---:|
| 1 | 256, 128 | 0.2 | 1e-3 | 0.7489 | 0.2097 |
| 2 | 512, 256, 128 | 0.3 | 1e-3 | 0.7489 | 0.2101 |
| 3 | 128, 64 | 0.1 | 1e-3 | 0.7487 | 0.2081 |
| 4 | 256, 128 | 0.2 | 3e-4 | 0.7494 | 0.2098 |

LightGBM: 0.7553 and 0.2138. Network size, dropout and learning rate moved PR-AUC by under 0.001 for adverse_outcome and by 0.002 for readmission_30d.

## Result: MLP rejected for both targets
- adverse_outcome (setting 4): seeds 0.7494, 0.7494, 0.7495. 3-seed ensemble 0.7501. Ensemble minus LightGBM: -0.0051 (95% interval -0.0059 to -0.0044). Above Logistic Regression by +0.0037 (0.0030 to 0.0044).
- readmission_30d (setting 2): seeds 0.2101, 0.2102, 0.2115. Ensemble 0.2123. Ensemble minus LightGBM: -0.0015 (-0.0029 to +0.0001). Below Logistic Regression by -0.0025 (-0.0035 to -0.0011).
- All three rules fail on both targets. Seed SD is 0.0000 to 0.0008, much smaller than the gaps, so the verdict does not depend on seed luck.
- The MLP is recorded as a rejected challenger. LightGBM stays the selected model for both targets.

## Limitations
- Four settings of one architecture family. This does not show that no neural model could match LightGBM.
- Early stopping on val gave the MLP the same advantage as the boosted models.
- The 3-seed ensemble is a diagnostic. The decision uses per-seed values and the paired interval.
- Bootstrap intervals measure sampling noise on val 2022 only.

## Moves to Notebook 06
- Same dev and val, same 90 features, same metrics, same early-stopping rule.
- The TabTransformer must pass the same three-rule test against LightGBM, and its result goes beside the MLP's.
- Load data/interim/val_predictions_classical.parquet (and val_predictions_mlp.parquet) for paired comparisons.
- Budget: 3 settings per target, then 3 seeds of the best one. Fewer settings because each run costs more.